# Plot Ridge Decoder Performance vs Neuron Subset Size Color-Coded by Recording Depth

This notebook loads the neuron subset decoder results (`ridge_decoder_neuron_subsets.parquet`) for all V1 sessions and plots the decoder performance (Mean $R^2$ and Mean Absolute Error) as a function of the subset size, color-coded by the nominal imaging depth of the recording (obtained from the flexilims session entity).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

import seaborn as sns
from pathlib import Path
from cottage_analysis.summary_analysis import get_session_list

# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
# Set it as the default font family
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {"PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"}

In [ ]:
from cottage_analysis.summary_analysis import load_project_subsets

# Set cut_treadmill to True to load the version with cut acceleration/transient periods,
# or False to load the no-cut version.
cut_treadmill = True

# 1. Load data for each project separately
df_ccyp = load_project_subsets(
    "ccyp_l5_3d_vision",
    session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
    cut_treadmill=cut_treadmill
)
df_colasa = load_project_subsets(
    "colasa_3d-vision_revisions",
    session_to_exclude=SESSIONS_TO_EXCLUDE.keys(),
    cut_treadmill=cut_treadmill
)
# 2. Add project identifiers
df_ccyp["project"] = "ccyp_l5_3d_vision"
df_colasa["project"] = "colasa_3d-vision_revisions"

# 3. Combine them
subsets_all_df = pd.concat([df_ccyp, df_colasa], ignore_index=True)

In [ ]:
if not subsets_all_df.empty:
    print("Targets:", subsets_all_df["target"].unique())
    print("Conditions:", subsets_all_df["condition"].unique())
    print("Nominal depths:", sorted(subsets_all_df["nominal_depth"].dropna().unique()))
    

In [ ]:
KEEP_NA = False

# Determine depth bounds for colormap
depths = sorted(subsets_all_df["nominal_depth"].dropna().unique())
norm = plt.Normalize(vmin=min(depths), vmax=max(depths))
cmap = plt.cm.viridis

fig = plt.figure(figsize=(18, 5))
axes = [fig.add_axes([0.01 + 0.3 * i, 0.05, 0.27, 0.9]) for i in range(3)]
    
fig.suptitle("Ridge Decoder Performance (Mean $R^2$) - Treadmill/Motor Data", y=1.05, fontsize=16)

target_mapping = {
    "OF_stim": ("Optic Flow (OF)", axes[0]),
    "RS_stim": ("Running Speed (RS)", axes[1]),
    "depth": ("Depth", axes[2])
}

# Usually for treadmill it is closedloop
cond = "closedloop"

for target, (title, ax) in target_mapping.items():
    cond_df = subsets_all_df[(subsets_all_df["target"] == target) & (subsets_all_df["condition"] == cond)]
    
    # Plot each session's curve
    for session_name, sess_df in cond_df.groupby("session"):
        sess_df = sess_df.sort_values("subset_size")
        depth = sess_df["nominal_depth"].iloc[0]
        if pd.isna(depth):
            if KEEP_NA:
                color = "gray"
            else:
                continue
        else:
            color = cmap(norm(depth))
        
        ax.plot(sess_df["subset_size"], sess_df["r2_mean"], marker='o', color=color, alpha=0.75, linewidth=2)
        ax.fill_between(
            sess_df["subset_size"], 
            sess_df["r2_mean"] - sess_df["r2_std"], 
            sess_df["r2_mean"] + sess_df["r2_std"], 
            color=color, alpha=0.08
        )
        
    ax.set_title(title)
    ax.set_xlabel("Subset Size (Neurons)")
    if ax == axes[0]:
        ax.set_ylabel("Mean $R^2$")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_ylim(0, 0.9)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cax = fig.add_axes([0.9, 0.1, 0.01, 0.7])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical', shrink=0.8, aspect=20)
cbar.set_label('Recording Depth (µm)')

plt.tight_layout()
plt.show()


In [ ]:
if not subsets_all_df.empty and "mae_mean" in subsets_all_df.columns:
    KEEP_NA = False
    depths = sorted(subsets_all_df["nominal_depth"].dropna().unique())
    norm = plt.Normalize(vmin=min(depths), vmax=max(depths))
    cmap = plt.cm.viridis
    
    fig = plt.figure(figsize=(18, 5))
    axes = [fig.add_axes([0.01 + 0.3 * i, 0.05, 0.27, 0.9]) for i in range(3)]
    fig.suptitle("Ridge Decoder Mean Absolute Error (MAE) - Treadmill/Motor Data", y=1.05, fontsize=16)
    
    target_mapping = {
        "OF_stim": ("Optic Flow (OF)", axes[0]),
        "RS_stim": ("Running Speed (RS)", axes[1]),
        "depth": ("Depth", axes[2])
    }
    
    cond = "closedloop"
    
    for target, (title, ax) in target_mapping.items():
        cond_df = subsets_all_df[(subsets_all_df["target"] == target) & (subsets_all_df["condition"] == cond)]
        
        # Plot each session's curve
        for session_name, sess_df in cond_df.groupby("session"):
            sess_df = sess_df.sort_values("subset_size")
            depth = sess_df["nominal_depth"].iloc[0]
            if pd.isna(depth):
                if KEEP_NA:
                    color = "gray"
                else:
                    continue
            else:
                color = cmap(norm(depth))
            
            ax.plot(sess_df["subset_size"], sess_df["mae_mean"], marker='o', color=color, alpha=0.75, linewidth=2)
            ax.fill_between(
                sess_df["subset_size"], 
                sess_df["mae_mean"] - sess_df["mae_std"], 
                sess_df["mae_mean"] + sess_df["mae_std"], 
                color=color, alpha=0.08
            )
            
        ax.set_title(title)
        ax.set_xlabel("Subset Size (Neurons)")
        if ax == axes[0]:
            ax.set_ylabel("Mean MAE")
        ax.grid(True, linestyle="--", alpha=0.5)
        
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cax = fig.add_axes([0.9, 0.1, 0.01, 0.7])
    cbar = fig.colorbar(sm, cax=cax, orientation='vertical', shrink=0.8, aspect=20)
    cbar.set_label('Recording Depth (µm)')
    
    plt.tight_layout()
    plt.show()


In [ ]:
subsets_all_df

In [ ]:
# Now split population at 350um and plot the rsquare for the 3 decoders on the same axis
cut_off = 350

fig, ax = plt.subplots(figsize=(10, 7))
target_mapping = {
    "OF_stim": "Optic Flow (OF)",
    "RS_stim": "Running Speed (RS)",
    "depth": "Depth"
}

# Optional color mapping to keep targets color-consistent
colors = {
    "OF_stim": "C0",
    "RS_stim": "C1",
    "depth": "C2"
}

cond = "closedloop"
max_sizes = subsets_all_df.groupby("session")["subset_size"].transform("max")
filtered_subsets_df = subsets_all_df[subsets_all_df["subset_size"] < max_sizes]

for target, title in target_mapping.items():
    cond_df = filtered_subsets_df[(filtered_subsets_df["target"] == target) & (filtered_subsets_df["condition"] == cond)]
    color = colors[target]
    
    # --- Layer 2/3 (depth <= cut_off) -> Circles ---
    layer23 = cond_df[cond_df.nominal_depth <= cut_off]
    if not layer23.empty:
        avg_l23 = layer23.groupby('subset_size')['r2_mean'].mean()
        ax.plot(avg_l23.index, avg_l23.values, label=f"{title} (L2/3)", marker='o', color=color, linewidth=2)
        
    # --- Layer 5 (depth > cut_off) -> Squares ---
    layer5 = cond_df[cond_df.nominal_depth > cut_off]
    if not layer5.empty:
        avg_l5 = layer5.groupby('subset_size')['r2_mean'].mean()
        ax.plot(avg_l5.index, avg_l5.values, label=f"{title} (L5)", marker='s', color=color, linewidth=2, linestyle='--')

# --- Label axis and clean up styling ---
ax.set_title(rf"Decoder Performance by Layer (Cut-off: {cut_off} µm)")
ax.set_xlabel("Subset Size")
ax.set_ylabel("Mean $R^2$")
ax.set_xscale("log")  # Using log scale since subset sizes typically scale exponentially
ax.legend(frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
subset200 = subsets_all_df[subsets_all_df.subset_size==200]
table_mae = subset200.pivot(
    index=['session', 'nominal_depth'], 
    columns='target', 
    values='mae_mean'
).reset_index()

target_mapping = {
    "OF_stim": "Optic Flow (OF)",
    "RS_stim": "Running Speed (RS)",
    "depth": "Depth"
}

fig, axes = plt.subplots(1,3, figsize=(14,5))
pairs = [
    ('OF_stim', 'RS_stim'),
    ('OF_stim', 'depth'),
    ('depth', 'RS_stim')
]
for i, (x_col, y_col) in enumerate(pairs):
    ax = axes[i]
    
    # Generate scatter plot color-coded by nominal depth
    scatter = ax.scatter(
        table_mae[x_col],
        table_mae[y_col],
        c=table_mae['nominal_depth'],
        cmap='viridis',
        s=60,             # Marker size
        edgecolors='none',
        alpha=0.8
    )
    
    # Label the axes and set titles
    ax.set_xlabel(f"MAE {target_mapping[x_col]}")
    ax.set_ylabel(f"MAE {target_mapping[y_col]}")
    
    # Adjust layout ticks/spines for a clean style
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
# 4. Add a unified colorbar on the right side of the figure
cbar = fig.colorbar(scatter, ax=axes.ravel().tolist(), pad=0.03, aspect=20)
cbar.set_label("Nominal Depth (µm)")


In [ ]:
subset200 = subsets_all_df[subsets_all_df.subset_size==200]
table_mae = subset200.pivot(
    index=['session', 'nominal_depth'], 
    columns='target', 
    values='r2_mean'
).reset_index()

target_mapping = {
    "OF_stim": "Optic Flow (OF)",
    "RS_stim": "Running Speed (RS)",
    "depth": "Depth"
}

fig, axes = plt.subplots(1,3, figsize=(14,5))
pairs = [
    ('OF_stim', 'RS_stim'),
    ('OF_stim', 'depth'),
    ('depth', 'RS_stim')
]
for i, (x_col, y_col) in enumerate(pairs):
    ax = axes[i]
    
    # Generate scatter plot color-coded by nominal depth
    scatter = ax.scatter(
        table_mae[x_col],
        table_mae[y_col],
        c=table_mae['nominal_depth'],
        cmap='viridis',
        s=60,             # Marker size
        edgecolors='none',
        alpha=0.8
    )
    
    # Label the axes and set titles
    ax.set_xlabel(f"$r^2$ mean {target_mapping[x_col]}")
    ax.set_ylabel(f"$r^2$ mean {target_mapping[y_col]}")
    
    # Adjust layout ticks/spines for a clean style
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
# 4. Add a unified colorbar on the right side of the figure
cbar = fig.colorbar(scatter, ax=axes.ravel().tolist(), pad=0.03, aspect=20)
cbar.set_label("Nominal Depth (µm)")
